In [1]:
import pandas as pd

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 100)        # Set the display width to 1000 characters
pd.set_option('display.max_colwidth', None) # Allow the full content of each column to be displayed

In [2]:
df_final_audit = pd.read_csv('./comprehensive_audit_final.csv')

In [11]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── 데이터 로드 ──────────────────────────────────────────
df = pd.read_csv('governance_audit_full.csv', encoding='utf-8-sig')
print(f"데이터: {len(df)}건")

# ── GRI 직접 사용 / 음수 부동소수점 오차 보정 ─────────────
gri = df['GRI'].clip(lower=0)
print(f"GRI 컬럼 직접 사용 (N={len(gri)}건)")

# ── 이론적 최대값 산출 ────────────────────────────────────
# F_risk 최대 = 1 (Sf=0), U_risk 최대 = 1 (Cr=0)
# Conflict 최대 = 2등급 차이 * 0.5 = 1.0 → 가중치 0.2 적용 시 0.2
gri_theoretical_max = 0.4 * 1.0 + 0.4 * 1.0 + 0.2 * 1.0
print(f"이론적 최대값: {gri_theoretical_max:.2f}")

# ── 분포 통계량 산출 ─────────────────────────────────────
print("\n" + "=" * 60)
print("GRI 분포 통계량 (N = 10,058)")
print("=" * 60)
print(f"이론적 최대값    : {gri_theoretical_max:.4f}")
print(f"실제 최대값      : {gri.max():.4f}")
print(f"실제 최솟값      : {gri.min():.4f}")
print(f"평균             : {gri.mean():.4f}")
print(f"중앙값(P50)      : {gri.median():.4f}")
print(f"표준편차         : {gri.std():.4f}")
print(f"왜도(Skewness)   : {stats.skew(gri):.4f}")
print(f"첨도(Kurtosis)   : {stats.kurtosis(gri):.4f}")
print(f"P25              : {gri.quantile(0.25):.4f}")
print(f"P75              : {gri.quantile(0.75):.4f}")
print(f"IQR              : {gri.quantile(0.75) - gri.quantile(0.25):.4f}")
print(f"P90              : {gri.quantile(0.90):.4f}")
print(f"P95              : {gri.quantile(0.95):.4f}")
print(f"P99              : {gri.quantile(0.99):.4f}")

# ── 구간별 분포 ──────────────────────────────────────────
print("\n" + "=" * 60)
print("GRI 구간별 분포")
print("=" * 60)
bins = [0, 0.05, 0.10, 0.18, gri.max() + 0.001]
labels = ['L1(<0.05)', 'L2(0.05~0.10)', 'L3(0.10~0.18)', 'L4(≥0.18)']
gri_cat = pd.cut(gri, bins=bins, labels=labels, right=False)
dist = gri_cat.value_counts().sort_index()
for label, cnt in dist.items():
    print(f"  {label}: {cnt}건 ({cnt/len(gri)*100:.1f}%)")

# ── GRI=0 비율 (완전 자동화 핵심) ────────────────────────
print(f"\nGRI = 0 (완전 무결성·합의·일치): {(gri == 0).sum()}건 "
      f"({(gri == 0).mean()*100:.1f}%)")
print(f"GRI > 0.5 (고위험):              {(gri > 0.5).sum()}건 "
      f"({(gri > 0.5).mean()*100:.1f}%)")

# ── 저장 ─────────────────────────────────────────────────
summary = {
    '이론적최대값': gri_theoretical_max,
    '실제최대값': gri.max(), '실제최솟값': gri.min(),
    '평균': gri.mean(), '중앙값': gri.median(),
    '표준편차': gri.std(), '왜도': stats.skew(gri),
    '첨도': stats.kurtosis(gri),
    'P25': gri.quantile(0.25), 'P75': gri.quantile(0.75),
    'P90': gri.quantile(0.90), 'P95': gri.quantile(0.95),
    'P99': gri.quantile(0.99)
}
pd.DataFrame([summary]).to_csv('gri_distribution_result.csv',
                                index=False, encoding='utf-8-sig')
print("\n결과 저장: gri_distribution_result.csv")

데이터: 10058건
GRI 컬럼 직접 사용 (N=10058건)
이론적 최대값: 1.00

GRI 분포 통계량 (N = 10,058)
이론적 최대값    : 1.0000
실제 최대값      : 0.6600
실제 최솟값      : 0.0000
평균             : 0.0487
중앙값(P50)      : 0.0000
표준편차         : 0.0733
왜도(Skewness)   : 1.5021
첨도(Kurtosis)   : 2.1181
P25              : 0.0000
P75              : 0.1000
IQR              : 0.1000
P90              : 0.1600
P95              : 0.1800
P99              : 0.2600

GRI 구간별 분포
  L1(<0.05): 6445건 (64.1%)
  L2(0.05~0.10): 2122건 (21.1%)
  L3(0.10~0.18): 1042건 (10.4%)
  L4(≥0.18): 449건 (4.5%)

GRI = 0 (완전 무결성·합의·일치): 5835건 (58.0%)
GRI > 0.5 (고위험):              1건 (0.0%)

결과 저장: gri_distribution_result.csv
